# Assignment: Capacity under Demand Uncertainty

**Course:** Optimization of Energy Systems (15s09)  
**Authors:** Max Glanville & Andres Garcia Angulo  
**Stack:** Julia · JuMP · Gurobi

A hyperscaler must choose a grid connection offer and optional onsite gas capacity **before** knowing AI compute demand and revenue. This notebook scaffolds a two-stage stochastic MILP for that decision.


## Learning goals

By the end of this assignment you should be able to:

1. Frame a hyperscaler power-procurement problem as a **two-stage stochastic MILP** (here-and-now vs wait-and-see).
2. Encode **firm vs flexible** grid offers and optional **onsite CCGT** capacity in JuMP.
3. Use **mock campus / workload / plant data**, then replace it with real sources (PJM, Nvidia, EIA, etc.).
4. Compare contracts on **expected NPV** and probe thresholds (curtailment severity, revenue per MW, workload mix).
5. State clearly what the model leaves out (batteries, endogenous prices, GPU depreciation, …).


## Problem statement

A hyperscaler wants its campus earning revenue as soon as possible, but the grid cannot supply firm power quickly. It can:

- connect sooner if it agrees to **cut consumption in peak hours** (flexible / curtailable), or
- wait for **firm** grid service, and/or
- **build onsite generation** (expensive; may sit idle much of the year).

Both grid and generation commitments are made years ahead and are hard to reverse, while the value of compute is uncertain (workload mix, utilisation, chip efficiency).

**Research question:** How should a profit-maximising hyperscaler secure power for a new AI data-centre campus (build onsite generation, accept a curtailable grid connection, wait for firm grid service, or a mix) when it must commit before it knows how much computing work will arrive, what kind, and what it will earn?


## Environment & Gurobi note

Activate this folder’s project, then load JuMP. **Gurobi** needs a license (`gurobi.lic` / academic license). If Gurobi is unavailable, complete Tasks A–B and skip solve cells — note that in the checklist.


In [ ]:
using Pkg
Pkg.activate(".")
# Pkg.instantiate()  # after Manifest.toml exists
# Pkg.add(["JuMP", "Gurobi", "DataFrames", "CSV"])

using JuMP
# using Gurobi   # uncomment when licensed Gurobi is installed
using DataFrames

println("Julia ", VERSION)
println("JuMP loaded. Uncomment Gurobi when ready to solve.")


## Mock data (stand-in for appendix tables)

Standard campus: **100 MW IT load**, ~current-generation Nvidia Blackwell-class racks. Two grid offers: **K1 Firm** (later connection, no curtailment) and **K2 Flexible** (earlier connection, peak curtailment). Demand scenarios ω differ by volume / mix multipliers θ_ω with probabilities p_ω.

> Replace these placeholders with real offers and parameters when you complete Task A.


In [ ]:
# --- Sets (toy) ---
K = [:firm, :flexible]          # grid offers
J = [:inference, :agentic, :training]  # workload classes
Ω = [:low, :base, :high]        # demand scenarios
T = 1:24                        # representative hours (extend later)

# --- Grid offers (mock; inspired by proposal Table 2) ---
grid = Dict(
    :firm => (
        lead_years = 4.0,
        curtail_frac = 0.0,          # fraction of hours curtailed
        capacity_mw = 100.0,
        fixed_charge_m_per_year = 12.0,  # $m / year (placeholder)
        energy_price = 45.0,         # $/MWh
    ),
    :flexible => (
        lead_years = 1.5,
        curtail_frac = 0.15,
        capacity_mw = 100.0,
        fixed_charge_m_per_year = 6.0,
        energy_price = 40.0,
    ),
)

# --- Campus (mock Table 3) ---
campus = (
    it_load_mw = 100.0,
    n_gpus = 58_800,
    pue = 1.25,                     # facility power / IT power
)

# --- Workload / demand uncertainty (mock Table 4) ---
p_ω = Dict(:low => 0.25, :base => 0.50, :high => 0.25)
θ_ω = Dict(:low => 0.70, :base => 1.00, :high => 1.30)  # demand multiplier
rev_per_mw_year = Dict(             # $m / MW-IT / year if fully utilised
    :inference => 11.0,
    :agentic => 10.0,
    :training => 9.0,
)
flex_share = Dict(                  # share that can shift off peak
    :inference => 0.20,
    :agentic => 0.40,
    :training => 0.80,
)

# --- Onsite CCGT (mock Table 5) ---
ccgt = (
    capex_per_mw = 1.2e6,           # $ / MW installed
    om_fixed_per_mw_year = 2.0e4,
    heat_rate = 7.0,                # MMBtu / MWh
    fuel_price = 3.5,               # $ / MMBtu
    avail = 0.92,
    life_years = 20,
)

discount_rate = 0.08
horizon_years = 10

DataFrame(
    offer = collect(keys(grid)),
    lead_years = [grid[k].lead_years for k in keys(grid)],
    curtail_frac = [grid[k].curtail_frac for k in keys(grid)],
)


## Model sketch (two-stage stochastic MILP)

### Stage 1 (here-and-now) — before ω is revealed
- Binary (or SOS1) choice of grid offer: \(z_k\), \(\sum_k z_k = 1\).
- Continuous onsite capacity: \(g \ge 0\) (MW of CCGT).

### Stage 2 (wait-and-see) — for each scenario ω, hour t, workload j
- Work processed, grid import, turbine dispatch, unmet / deferred demand, …

### Objective
Maximise **expected discounted profit** (compute revenue − energy − fuel − O&M − annualised capex − grid fixed charges), accounting for lead time before the campus is energised.

### Constraint families (proposal C1–C10)
- Work accounting & GPU capacity caps  
- Power balance (IT × PUE)  
- Turbine limits ≤ built capacity × availability  
- Grid imports only after connection and within offer rules (curtailment windows for flexible)  
- Logic linking \(z_k\) to which offer parameters apply  

**Problem class:** two-stage stochastic **MILP** (integer only because of discrete contract choice \(z\)). Solver: **Gurobi via JuMP**.


## Task A — Data TODOs

Fill or replace the mock tables. For each item: *what you need*, *why*, *source you will use*.

| ID | TODO | Status |
|----|------|--------|
| A1 | Real grid offers (TX / VA / OR or PJM-style): lead time, curtailment rules, capacity, charges | `[ ]` |
| A2 | Campus IT load, GPU count, PUE, revenue per MW by workload | `[ ]` |
| A3 | Demand scenarios θ_ω, p_ω, and workload mix by ω | `[ ]` |
| A4 | CCGT capex, heat rate, fuel, O&M, lifetime (EIA / OEM) | `[ ]` |
| A5 | Peak-hour mask for flexible offer (which hours curtailed) | `[ ]` |
| A6 | Document units and USD year for every money parameter | `[ ]` |

### Deliverable
Update the dictionaries above (or load CSVs) and briefly cite sources in a markdown cell under this section.


In [ ]:
# Task A helpers — load CSVs once you create them, e.g.:
# using CSV
# offers = CSV.read("data/grid_offers.csv", DataFrame)
# Write your data-cleaning / validation code here.

function assert_probabilities(p::Dict)
    s = sum(values(p))
    @assert abs(s - 1.0) < 1e-8 "Scenario probabilities must sum to 1 (got $s)"
end

assert_probabilities(p_ω)
println("Mock probabilities OK. Replace with real data for Task A.")


## Task B — JuMP stubs (two-stage MILP)

Build the model structure. Keep stage-1 decisions shared across scenarios; stage-2 indexed by ω (and t, j as needed).

> Stub only: complete formulations are your work. Do not solve until Task C.


In [ ]:
"""
build_hyperscaler_model(; optimizer)

Returns a JuMP model with:
  Stage 1: z[k] binary grid-offer choice, g >= 0 onsite MW
  Stage 2: placeholders for dispatch / work variables by ω
  Objective: expected discounted profit (TODO)
  Constraints: stubs for C1–C10 (TODO)
"""
function build_hyperscaler_model(; optimizer = nothing)
    m = optimizer === nothing ? Model() : Model(optimizer)

    # --- Stage 1 ---
    @variable(m, z[K], Bin)                 # choose one grid offer
    @variable(m, g >= 0)                    # onsite CCGT capacity (MW)
    @constraint(m, choose_one_offer, sum(z[k] for k in K) == 1)

    # --- Stage 2 placeholders (expand with t, j as you refine) ---
    @variable(m, grid_mw[Ω, T] >= 0)        # grid import
    @variable(m, turb_mw[Ω, T] >= 0)        # turbine dispatch
    @variable(m, work[J, Ω, T] >= 0)        # work processed (define units!)

    # --- Linking stubs (TODO: real curtailment / lead-time logic) ---
    for ω in Ω, t in T
        # Power balance sketch: IT load served ≤ grid + turbine (scaled by PUE later)
        @constraint(m, turb_mw[ω, t] <= g * ccgt.avail)
        # Grid capped by selected offer capacity (big-M / convex combination TODO)
        @constraint(m, grid_mw[ω, t] <= sum(z[k] * grid[k].capacity_mw for k in K))
    end

    # --- Objective stub ---
    # Replace with E_ω[ NPV(revenue − costs | z, g, stage-2) ]
    @objective(m, Max, 0.0)

    return m
end

model = build_hyperscaler_model()
println(model)
println("Task B: flesh out objective + C1–C10; then pass Gurobi.Optimizer in Task C.")


## Task C — Solve

Once Gurobi is licensed and the formulation is complete:


In [ ]:
# Task C — solve (uncomment when ready)
# using Gurobi
# model = build_hyperscaler_model(optimizer = Gurobi.Optimizer)
# set_optimizer_attribute(model, "OutputFlag", 1)
# set_optimizer_attribute(model, "TimeLimit", 300)
# optimize!(model)
# println(termination_status(model))
# println("z = ", value.(model[:z]))
# println("g = ", value(model[:g]))

println("Task C: solve with Gurobi after Task B is complete.")


## Task D — NPV experiments & write-up

Experimental design from the proposal: compare **firm vs flexible** (and mixes with onsite gas) on expected NPV. Then stress key levers.

Suggested sweeps (start with one variable at a time):

1. Curtailment fraction / peak-hour severity on the flexible offer  
2. Revenue per MW-year  
3. Workload mix (more training vs inference)  
4. Fuel price or CCGT capex  

**Null result to watch for:** firm always beats flexible in every scenario (or the reverse).

### Write-up (markdown below or separate `writeup.md`)
- Optimal stage-1 plan and expected NPV  
- NPV by scenario ω  
- Thresholds where the preferred contract flips  
- ½–1 page interpretation tied to the research question


In [ ]:
# Task D — experiment harness (fill in once model solves)
function expected_npv_stub(z_choice::Symbol, g_mw::Float64)
    # TODO: evaluate or re-solve with z fixed / g fixed and return E[NPV]
    return NaN
end

results = DataFrame(
    offer = Symbol[],
    g_mw = Float64[],
    expected_npv = Float64[],
)
for k in K, g in (0.0, 50.0, 100.0)
    push!(results, (k, g, expected_npv_stub(k, g)))
end
results


### Task D write-up space

*Replace this cell with your findings.*


## Task E — Stretch

Pick **one** (or propose your own) and sketch how you would extend the JuMP model:

- Endogenous **electricity prices** if many campuses connect (not treated as fixed inputs)  
- **Batteries / solar** or multi-site workload shifting  
- **GPU choice** / efficiency as a stage-1 decision  
- Risk measure beyond expected profit (CVaR, worst-case ω)  
- Replace mock offers with a **panel of real US interconnection options**


In [ ]:
# Task E notes / prototype hooks
stretch_ideas = [
    "CVaR on NPV across Ω",
    "Battery SoC balance + charge/discharge vars",
    "Binary GPU portfolio with different J efficiencies",
]
stretch_ideas


## Out of scope (from the proposal)

Be explicit in your write-up that the baseline model **does not** include:

- Demand-driven transmission upgrades or equilibrium prices  
- Uncertain / stochastic curtailment timing (we assume peak windows known ex ante)  
- Batteries, solar, or spatial flexibility across sites  
- GPU depreciation or efficiency drift over the horizon  
- Over-weighting of extreme scenarios for commercial risk appetite


## Checklist

- [ ] Learning goals read; research question stated in own words  
- [ ] Env activates; JuMP imports (Gurobi noted if blocked)  
- [ ] Task A data TODOs addressed or explicitly deferred with sources  
- [ ] Task B JuMP two-stage stubs filled toward a solvable MILP  
- [ ] Task C solve (or documented license blocker)  
- [ ] Task D NPV table + short write-up  
- [ ] Task E stretch idea noted  
- [ ] Out-of-scope limitations listed  
- [ ] `git push` your work to this repo
